**FASE 5b: DIAGNÓSTICO GEOMÉTRICO Y OPTIMIZACIÓN CONJUNTA (GRID SEARCH)**

Autor: Andoni Cabrera Fernández

Descripción: Este cuaderno carga la telemetría topológica tridimensional (EAR y MAR) serializada en los archivos `.pkl`.
Implementa la lógica estricta descrita en la memoria:
1. **Calibración Biométrica Asíncrona:** Periodo de *Warm-up* dinámico para extraer $\mu$ y $\sigma$ del EAR, estableciendo un umbral de cierre adaptado a la fisionomía del usuario con un límite duro de seguridad ($0.18$).
2. **Late Fusion Multiclase:** Diagnóstico de los estados KSS (0, 5, 10) priorizando el PERCLOS para somnolencia crítica y el bostezo ponderado para la baja vigilancia.

Se ejecuta un Grid Search exhaustivo evaluando el rendimiento en función de los FPS, los umbrales fisiológicos y la duración del periodo de calibración.

**1. IMPORTACIÓN E INFRAESTRUCTURA**

In [ ]:
import pickle
import pandas as pd
import numpy as np
import time
from sklearn.metrics import accuracy_score, f1_score
from google.colab import drive

drive.mount('/content/drive')
ruta_base = '/content/drive/MyDrive/TFG_Fatiga_Colab/PKL_GEOMETRIA/'

# Cargar los 3 diccionarios de geometría a DataFrames
print("Cargando bases de datos geométricas desde la memoria local...")
with open(f'{ruta_base}geometria_5fps.pkl', 'rb') as f: df_5 = pd.DataFrame(pickle.load(f))
with open(f'{ruta_base}geometria_15fps.pkl', 'rb') as f: df_15 = pd.DataFrame(pickle.load(f))
with open(f'{ruta_base}geometria_30fps.pkl', 'rb') as f: df_30 = pd.DataFrame(pickle.load(f))

print(f"Datos cargados. Registros procesados: 5FPS ({len(df_5)}), 15FPS ({len(df_15)}), 30FPS ({len(df_30)})")

Mounted at /content/drive
Cargando bases de datos geométricas desde la memoria local...


**2. MOTOR MATEMÁTICO: CALIBRACIÓN ASÍNCRONA Y DIAGNÓSTICO MULTICLASE**

In [ ]:
import time
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# =========================================================
# LA FUNCIÓN EVALUADORA ABSOLUTA (V5 - Geometría + Biología Clínica)
# =========================================================
def evaluar_videos_geometricos_definitivo(df, fps, warmup_sec=20,
                                          perclos_th=0.10, perclos_window_sec=60,
                                          microsleep_sec_th=0.6,
                                          bostezo_sec_th=1.0,
                                          bpm_max_th=28, duracion_media_parpadeo_th=0.4,
                                          sigma_mult_ear=2.0, ear_min=0.18,
                                          sigma_mult_mar=3.0, mar_min=0.25,
                                          smooth_frames=3):
    resultados = []
    for video_name, grupo in df.groupby('video'):
        total_frames = len(grupo)
        if total_frames == 0: continue
        clase_real = grupo['clase_real'].iloc[0]

        frames_warmup = int(warmup_sec * fps)
        if frames_warmup == 0 or frames_warmup > total_frames:
            frames_warmup = min(20 * fps, total_frames)

        # 1. SUAVIZADO DE SEÑAL (Anti-Jitter de MediaPipe)
        ear_suave = grupo['ear'].rolling(window=smooth_frames, min_periods=1).mean()
        mar_suave = grupo['mar'].rolling(window=smooth_frames, min_periods=1).mean()

        # 2. CALIBRACIÓN ROBUSTA (Usando Percentiles)
        ear_calibracion = ear_suave.iloc[:frames_warmup]
        base_ear = ear_calibracion.quantile(0.90)  # Descarta parpadeos
        sigma_ear = ear_calibracion.std()
        umbral_ear_dinamico = max(base_ear - (sigma_mult_ear * sigma_ear), ear_min)

        mar_calibracion = mar_suave.iloc[:frames_warmup]
        base_mar = mar_calibracion.median()  # Postura relajada de los labios
        sigma_mar = mar_calibracion.std()
        umbral_mar_dinamico = max(base_mar + (sigma_mult_mar * sigma_mar), mar_min)

        # 3. ESTADO OCULAR (PERCLOS, Microsueños y Parpadeos)
        es_cerrado = ear_suave < umbral_ear_dinamico

        # 3.a) PERCLOS en Ventana Deslizante
        frames_ventana = int(perclos_window_sec * fps)
        if frames_ventana > total_frames or frames_ventana == 0:
            perclos_max = es_cerrado.sum() / total_frames
        else:
            perclos_rolling = es_cerrado.rolling(window=frames_ventana, min_periods=1).mean()
            perclos_max = perclos_rolling.max()

        # 3.b) Rachas y Parpadeos
        rachas_ojo = es_cerrado.groupby((~es_cerrado).cumsum()).sum()
        parpadeos_validos = rachas_ojo[rachas_ojo > 0]

        # Microsueño explícito
        max_frames_ojo_cerrado = parpadeos_validos.max() if not parpadeos_validos.empty else 0
        segundos_max_ojo_cerrado = max_frames_ojo_cerrado / fps

        # Duración media del parpadeo (Pesadez)
        duracion_media_parpadeo = (parpadeos_validos.mean() / fps) if not parpadeos_validos.empty else 0

        # Tasa de Parpadeo (BPM)
        transiciones_cierre = es_cerrado.astype(int).diff()
        num_parpadeos = (transiciones_cierre == 1).sum()
        minutos_video = total_frames / (fps * 60)
        bpm = num_parpadeos / minutos_video if minutos_video > 0 else 0

        # 4. ESTADO BUCAL (Bostezo Dinámico Sostenido)
        es_bostezo = mar_suave > umbral_mar_dinamico
        rachas_boca = es_bostezo.groupby((~es_bostezo).cumsum()).sum()
        segundos_max_bostezo = (rachas_boca.max() if not rachas_boca.empty else 0) / fps

        # 5. ÁRBOL DE DECISIÓN CLÍNICO
        if perclos_max >= perclos_th or segundos_max_ojo_cerrado >= microsleep_sec_th:
            clase_predicha = 10  # DORMIDO (PERCLOS alto o cabezada/microsueño)
        elif segundos_max_bostezo >= bostezo_sec_th or duracion_media_parpadeo >= duracion_media_parpadeo_th or bpm >= bpm_max_th:
            clase_predicha = 5   # FATIGA MODERADA (Bostezo genuino, parpadeo lento o sequedad ocular)
        else:
            clase_predicha = 0   # ALERTA

        resultados.append({'video': video_name, 'real': clase_real, 'pred': clase_predicha})
    return pd.DataFrame(resultados)

**3. OPTIMIZACIÓN MULTIDIMENSIONAL (GRID SEARCH)**

**4. ANÁLISIS DETALLADO DEL MODELO ÓPTIMO (MATRIZ DE CONFUSIÓN Y CASOS)**

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# 1. Cargar la configuración óptima (La ganadora a 5 FPS)
fps_optimo = 5
df_data = df_5  # El DataFrame de 5 FPS cargado previamente
w_opt = 20.0
p_opt = 0.10
m_opt = 0.30
b_opt = 1.0

print(f"Evaluando modelo definitivo -> 5 FPS | W: {w_opt}s | P: {p_opt} | M: {m_opt} | B: {b_opt}s\n")
df_optimo = evaluar_videos_geometricos(
    df_data, fps=fps_optimo, warmup_sec=w_opt,
    perclos_th=p_opt, mar_th=m_opt, bostezo_sec_th=b_opt
)

# =========================================================
# 1. MATRIZ DE CONFUSIÓN
# =========================================================
cm = confusion_matrix(df_optimo['real'], df_optimo['pred'], labels=[0, 5, 10])

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['0 (Alerta)', '5 (Baja Vig.)', '10 (Dormido)'],
            yticklabels=['0 (Alerta)', '5 (Baja Vig.)', '10 (Dormido)'])
plt.xlabel('Predicción del Motor Geométrico')
plt.ylabel('Etiqueta Real (UTA-RLDD)')
plt.title('Matriz de Confusión - Fase 5 (TinyML Geométrico a 5 FPS)')
plt.show()

# =========================================================
# 2. REPORTE DE CLASIFICACIÓN (F1-Score por Clases)
# =========================================================
print("\n" + "="*50)
print("REPORTE DE CLASIFICACIÓN DETALLADO")
print("="*50)
print(classification_report(df_optimo['real'], df_optimo['pred'], labels=[0, 5, 10],
                            target_names=['Alerta (0)', 'Baja Vig. (5)', 'Dormido (10)'], zero_division=0))

# =========================================================
# 3. EXTRACCIÓN DE MEJORES Y PEORES CASOS
# =========================================================
aciertos = df_optimo[df_optimo['real'] == df_optimo['pred']]
fallos = df_optimo[df_optimo['real'] != df_optimo['pred']]

print("\n" + "="*50)
print("AUDITORÍA DE CASOS (MEJORES Y PEORES)")
print("="*50)

print("\nEJEMPLOS DE ACIERTOS (Casos de Éxito):")
# Mostramos 1 ejemplo de cada clase donde acertó
for clase in [0, 5, 10]:
    ejemplo = aciertos[aciertos['real'] == clase].head(1)
    if not ejemplo.empty:
        vid = ejemplo.iloc[0]['video']
        print(f" - El vídeo '{vid}' era Clase {clase} y se predijo correctamente como {clase}.")

print("\nEJEMPLOS DE FALLOS (El problema de la Clase 5):")
# Vamos a buscar los casos críticos de los que hablamos en la memoria
falsos_negativos = fallos[(fallos['real'] == 5) & (fallos['pred'] == 0)]
if not falsos_negativos.empty:
    print(f" - Peligro: El conductor estaba en Baja Vigilancia (5), pero la geometría dijo Alerta (0):")
    print(f"   Ejemplo -> Vídeo: {falsos_negativos.iloc[0]['video']}")

falsos_positivos_extremos = fallos[(fallos['real'] == 0) & (fallos['pred'] == 10)]
if not falsos_positivos_extremos.empty:
    print(f" - Exceso de Celo: El conductor estaba Alerta (0), pero la geometría detonó Dormido (10):")
    print(f"   Ejemplo -> Vídeo: {falsos_positivos_extremos.iloc[0]['video']}")